In [ ]:
from __future__ import annotations
 
import json
import time
from collections import OrderedDict
from pathlib import Path
from typing import Any
 
import torch
from torch import Tensor
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel
from safetensors.torch import save_file
 
from attribscope.data.trajectory import Trajectory, load_dataset
from attribscope.data.context   import (
    build_context_template,
    build_context_base,
    iter_scoreable_steps,
)

In [2]:
MODEL_PATH  = "/data/hoang/resources/models/meta-llama/Llama-3.1-8B-Instruct"
DATA_DIR    = "data/ww"
SUBSET      = "hand-crafted"       # subdirectory under DATA_DIR
OUTPUT_DIR  = "outputs/attn_mass/llama-3.1-8b/hand-crafted"
 
MAX_TOKENS  = 8192
CONTEXT     = "dependency"         # "dependency" | "all"
DTYPE       = "bfloat16"           # "bfloat16" | "float16" | "float32"
 
START_IDX   = 0
END_IDX     = None                 # None → all trajectories

In [3]:
device = torch.device("cuda:5")
dtype_map = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}
torch_dtype = dtype_map[DTYPE]
 
print(f"device: {device}  |  dtype: {DTYPE}")

device: cuda:5  |  dtype: bfloat16


In [4]:
print(f"Loading tokenizer: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
 
print(f"Loading model …")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch_dtype,
    device_map={"": device},
    # attn_implementation="eager",
)
model.eval()
 
n_layers = model.config.num_hidden_layers
n_heads  = model.config.num_attention_heads
print(f"  layers: {n_layers}  |  heads: {n_heads}  |  "
      f"params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Loading tokenizer: /data/hoang/resources/models/meta-llama/Llama-3.1-8B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model …


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  layers: 32  |  heads: 32  |  params: 8.03B


In [5]:
trajectories: list[Trajectory] = load_dataset(DATA_DIR, subset=SUBSET)
end_idx       = END_IDX if END_IDX is not None else len(trajectories)
trajectories  = trajectories[START_IDX:end_idx]
print(f"Loaded {len(trajectories)} trajectories [{START_IDX}:{end_idx}]")
 
# Peek at the first trajectory
traj0 = trajectories[0]
print(f"  Example: {traj0.filename}  |  {len(traj0.history)} steps  |  "
      f"mistake @ step {traj0.mistake_step}")

Loaded 58 trajectories [0:58]
  Example: 1.json  |  29 steps  |  mistake @ step 12


In [7]:
from attribscope.data.context import select_context, _serialize_turns
from transformers import PreTrainedTokenizer
def preprocess_context(
    traj:       Trajectory,
    step_idx:   int,
    tokenizer:  PreTrainedTokenizer,
    max_tokens: int | None = None,
    strategy:   str = "dependency",
) -> dict[str, Any]:

    history      = traj.history
    ctx_indices  = select_context(history, step_idx, strategy=strategy)
    step_content = _serialize_turns(history, [step_idx])
    assistant_msg = {"role": "assistant", "content": step_content}

    def _apply(indices: list[int]) -> tuple:
        """Tokenise [user_msg, assistant_msg] and the user-only prefix."""
        user_msg = {"role": "user", "content": _serialize_turns(history, indices)}
        full_ids = tokenizer.apply_chat_template(
            [user_msg, assistant_msg],
            tokenize              = True,
            add_generation_prompt = False,
            return_tensors        = "pt",
        )
        prefix_ids = tokenizer.apply_chat_template(
            [user_msg],
            tokenize              = True,
            add_generation_prompt = True,
            return_tensors        = "pt",
        )
        return full_ids, prefix_ids

    def _build_step_tokens(
        ctx_indices: list[int],
        ctx_len:     int,
        seq_len:     int,
    ) -> dict[int, list[int]]:
        """Map each step index to its token positions in full_ids.

        Context steps: found by progressive tokenization — the prefix grows
        one step at a time and the length delta gives the token span of each
        added step.

        Scored step (step_idx): always occupies [ctx_len, seq_len).
        """
        step_tokens: dict[int, list[int]] = {}

        # Compute prefix length after each cumulative prefix of ctx_indices.
        # prefix_lengths[k] = number of tokens in the prefix that contains
        # exactly the first k context steps.
        prefix_lengths = []
        for k in range(len(ctx_indices) + 1):
            _, partial_prefix = _apply(ctx_indices[:k])
            prefix_lengths.append(partial_prefix["input_ids"].shape[1])

        for k, idx in enumerate(ctx_indices):
            start = prefix_lengths[k]
            end   = prefix_lengths[k + 1]
            step_tokens[idx] = list(range(start, end))

        # The scored step always sits right after the context prefix.
        step_tokens[step_idx] = list(range(ctx_len, seq_len))

        return step_tokens

    full_ids, prefix_ids = _apply(ctx_indices)

    # ── Truncate context if full sequence exceeds max_tokens ─────────────
    if max_tokens is not None:
        while (
            full_ids["input_ids"].shape[1] > max_tokens
            and len(ctx_indices) > 0
        ):
            ctx_indices = ctx_indices[1:]   # drop oldest turn
            full_ids, prefix_ids = _apply(ctx_indices)

        if full_ids["input_ids"].shape[1] > max_tokens:
            # Hard truncation: step alone exceeds budget; slice from the front.
            # All ctx_indices have already been dropped, so step_tokens only
            # contains the scored step (no context step entries).
            step_len = full_ids["input_ids"].shape[1] - prefix_ids["input_ids"].shape[1]
            full_ids["input_ids"] = full_ids["input_ids"][:, -max_tokens:]
            ctx_len     = max(0, max_tokens - step_len)
            seq_len     = full_ids["input_ids"].shape[1]
            step_tokens = _build_step_tokens([], ctx_len, seq_len)
            return {
                "input_ids":   full_ids["input_ids"],
                "ctx_len":     ctx_len,
                "step_tokens": step_tokens,
            }

    ctx_len     = prefix_ids["input_ids"].shape[1]
    seq_len     = full_ids["input_ids"].shape[1]
    step_tokens = _build_step_tokens(ctx_indices, ctx_len, seq_len)

    return {
        "input_ids":   full_ids["input_ids"],
        "ctx_len":     ctx_len,
        "step_tokens": step_tokens,
    }

In [33]:
import torch
import torch.nn.functional as F
from attribscope.data.context import iter_scoreable_steps
import math

TEMPERATURE = 1.0               # softmax temperature
SIMILARITY  = "dot"          # "cosine" | "dot"
DIM         = 4096

results: dict[int, dict] = {}

for step_idx in tqdm(iter_scoreable_steps(traj0), desc="scoring steps"):
    encoded     = preprocess_context(
        traj0, step_idx, tokenizer,
        max_tokens=MAX_TOKENS, strategy=CONTEXT,
    )
    input_ids   = encoded["input_ids"].to(device)
    step_tokens = encoded["step_tokens"]

    # Context steps = everything in step_tokens except the scored step itself
    ctx_step_ids = sorted(m for m in step_tokens if m != step_idx)
    if not ctx_step_ids:
        continue                                                        # nothing to score against

    # ── Forward pass, collect hidden states ─────────────────────────────
    with torch.no_grad():
        out = model(input_ids, output_hidden_states=True, use_cache=False)
    hidden = torch.stack(out.hidden_states, dim=0).squeeze(1)           # (L+1, seq_len, d)
    del out

    # ── Mean-pool per step per layer ────────────────────────────────────  
    def pool(idxs: list[int]) -> Tensor:
        sel = torch.tensor(idxs, device=device, dtype=torch.long)
        return hidden[:, sel, :].mean(dim=1).float()                    # (L+1, d)

    h_i   = pool(step_tokens[step_idx])                                  # (L+1, d)
    h_ctx = torch.stack([pool(step_tokens[m]) for m in ctx_step_ids],
                        dim=0)                                           # (n_ctx, L+1, d)
    del hidden

    # ── Similarity per layer ────────────────────────────────────────────
    if SIMILARITY == "cosine":
        h_i   = F.normalize(h_i,   dim=-1)
        h_ctx = F.normalize(h_ctx, dim=-1)

    w_raw = torch.einsum("ld,nld->ln", h_i, h_ctx) / math.sqrt(DIM)                     # (L+1, n_ctx)

    # ── Softmax over context steps, within each layer ───────────────────
    weights = F.softmax(w_raw / TEMPERATURE, dim=-1)                     # (L+1, n_ctx)

    results[step_idx] = {
        "ctx_indices": ctx_step_ids,
        "weights":     weights.cpu(),
        "raw":         w_raw.cpu(),                                      # keep for inspection / temperature sweeps
    }

print(f"Computed causal weights for {len(results)} scored steps "
      f"| layers: {next(iter(results.values()))['weights'].shape[0]}")

scoring steps:   0%|          | 0/28 [00:00<?, ?it/s]

Computed causal weights for 28 scored steps | layers: 33


In [ ]:


from safetensors import safe_open
import safetensors.torch  # or safetensors.numpy, safetensors.flax

# Open and inspect a safetensors file
file_path = "/home/hoangpham/attribscope/outputs/weighting/llama-3.1-8b/hand-crafted/1.json.safetensors"

with safe_open(file_path, framework="pt", device="cpu") as f:
    # List all tensor keys
    print("Keys:", list(f.keys()))

    # Load a specific tensor
    tensor = f.get_tensor("21.raw_dot")
    print("Tensor shape:", tensor.shape)
    print("Tensor dtype:", tensor.dtype)

Keys: ['1.ctx_indices', '1.raw_cosine', '1.raw_dot', '10.ctx_indices', '10.raw_cosine', '10.raw_dot', '11.ctx_indices', '11.raw_cosine', '11.raw_dot', '12.ctx_indices', '12.raw_cosine', '12.raw_dot', '13.ctx_indices', '13.raw_cosine', '13.raw_dot', '14.ctx_indices', '14.raw_cosine', '14.raw_dot', '15.ctx_indices', '15.raw_cosine', '15.raw_dot', '16.ctx_indices', '16.raw_cosine', '16.raw_dot', '17.ctx_indices', '17.raw_cosine', '17.raw_dot', '18.ctx_indices', '18.raw_cosine', '18.raw_dot', '19.ctx_indices', '19.raw_cosine', '19.raw_dot', '2.ctx_indices', '2.raw_cosine', '2.raw_dot', '20.ctx_indices', '20.raw_cosine', '20.raw_dot', '21.ctx_indices', '21.raw_cosine', '21.raw_dot', '22.ctx_indices', '22.raw_cosine', '22.raw_dot', '23.ctx_indices', '23.raw_cosine', '23.raw_dot', '24.ctx_indices', '24.raw_cosine', '24.raw_dot', '25.ctx_indices', '25.raw_cosine', '25.raw_dot', '26.ctx_indices', '26.raw_cosine', '26.raw_dot', '27.ctx_indices', '27.raw_cosine', '27.raw_dot', '28.ctx_indices', '

In [30]:
import torch
import torch.nn.functional as F
from attribscope.data.context import iter_scoreable_steps
from safetensors.torch import load_file
import math

file_path = "/home/hoangpham/attribscope/outputs/weighting/llama-3.1-8b/hand-crafted/8.json.safetensors"

tensors = load_file(file_path)

In [89]:
w_raw[1, :]

tensor([0.8650, 0.9392, 0.9069, 0.9879, 0.9312, 0.9677, 0.9102])

In [87]:
TEMPERATURE = 0.1
DIM = 4096


w_raw = tensors['12.raw_cosine']
w_scaled = w_raw / (TEMPERATURE)
# w_scaled = w_raw
mass = F.softmax(w_scaled, dim=-1)

In [88]:
layer_idx = 18
w_raw[layer_idx], mass[layer_idx]

(tensor([0.4454, 0.6037, 0.5602, 0.8529, 0.5737, 0.7538, 0.5907]),
 tensor([0.0103, 0.0499, 0.0323, 0.6028, 0.0370, 0.2239, 0.0438]))

tensor([0.1413, 0.1416, 0.1423, 0.1456, 0.1425, 0.1446, 0.1419])